# CAP Theorem — Backend Interview Notes

---

## 1. Simple Intuition First

Imagine you and your friend maintain the **same notebook** but you live in different cities. To keep both copies in sync, you call each other every time either of you writes something new.

One day the phone line goes dead (network partition). Now you have two choices when someone asks "what's in the notebook right now?":

- **Wait until the phone line is back** to give a guaranteed correct answer → but until then, you can't answer at all. (Choosing **Consistency**)
- **Answer immediately with whatever you have**, even though it might be outdated compared to your friend's copy. (Choosing **Availability**)

You cannot do both while the line is dead. That's it. That's the entire theorem in one paragraph.

**One-liner to remember:** *"When the network breaks between nodes, you must pick between giving a correct answer (and possibly failing) or giving an answer (that might be stale)."*

Everything else in this doc is just formalizing and applying this idea to real backend systems.

---

## 2. Why This Topic Exists

- Modern backend systems don't run on a single DB server. Data is **replicated** across multiple nodes/data centers for scale and fault tolerance.
- Networks are **not reliable** — cables get cut, switches fail, cross-region latency spikes look like timeouts, a DC can go dark. This is not an edge case, it **will** happen at scale.
- CAP theorem gives you a **mental framework** to reason about what your system does *when* (not if) that failure happens.
- Companies use it to:
  - Choose the right database for the right use case (payments vs. product catalog vs. session store).
  - Design SLAs (what "available" and "correct" mean for a given service).
  - Make system design interview answers defensible ("I chose DynamoDB here because this data can tolerate eventual consistency, but I chose a CP store for the ledger").

This is why **every senior/staff backend and system design interview touches CAP** — it tests whether you can reason about failure, not just happy-path design.

---

## 3. Core Concepts

### The three properties
| Letter | Property | Meaning |
|---|---|---|
| **C** | Consistency | Every read returns the **most recent write**, or an error. (This is linearizability — NOT the "C" in ACID, don't confuse the two in interviews) |
| **A** | Availability | Every request gets a **non-error response** — no guarantee it's the latest data |
| **P** | Partition Tolerance | System keeps functioning even when nodes can't talk to each other |

### The real interview insight (most candidates get this wrong)
CAP is **not** "pick any 2 of 3." In a real distributed system, **network partitions will happen**, so Partition Tolerance isn't optional — you don't get to "give it up."

So the real choice is:
> **When a partition happens, do you sacrifice Consistency or Availability?**

That's why real systems are labeled **CP** or **AP** — not "CA" (CA only exists in theory, for a single-node or non-partition-able system, which isn't a real distributed system).

### PACELC (senior-level extension — commonly asked as a follow-up)
> **If** there's a **P**artition → choose **A**vailability or **C**onsistency.
> **Else** (normal operation, no partition) → choose **L**atency or **C**onsistency.

This matters because even when the network is healthy, syncing all replicas before responding (strong consistency) costs latency. This is why Cassandra/DynamoDB are fast even without a partition — they don't wait for every replica.

### Quorum math (very commonly asked, know this cold)
For a replication factor **N**, write quorum **W**, read quorum **R**:
- If **W + R > N** → every read is guaranteed to overlap with the latest write → strong consistency
- If **W + R ≤ N** → reads might miss the latest write → eventual consistency, but faster

This is exactly how Cassandra/DynamoDB let you **tune consistency per query** (`ONE`, `QUORUM`, `ALL`).

---

## 4. Production-Level Example

| System / Component | CAP Choice | Why |
|---|---|---|
| Payment ledger / bank balance | **CP** | Wrong balance is worse than a failed request |
| Inventory count for flash sale | **CP** (or carefully tuned) | Overselling stock = business/legal problem |
| Product catalog / listing pages | **AP** | Slightly stale price shown for 2 sec is fine, but page must load |
| Session store / auth tokens | **CP-leaning** | Stale "logged in" state is a security risk |
| Like/view counters, feeds | **AP** | Eventual consistency is totally fine |
| Zookeeper / etcd (service discovery, leader election) | **CP** | A stale leader pointer can cause split-brain — worse than unavailability |
| Cassandra / DynamoDB (used for logs, sessions at scale, IoT) | **AP** (tunable) | Massive write throughput, multi-region, tolerant of staleness |
| MongoDB (default majority write concern) | **CP** | Prioritizes correctness on primary failure |
| Redis Cluster | **CP-leaning** | Minority partition nodes stop serving writes to avoid split-brain |

**Realistic scenario to say out loud in an interview:**
"In a food delivery app (Zomato/Swiggy-style), the **order + payment service** must be CP — you cannot show a successful payment that didn't actually commit. But the **restaurant menu/ratings service** can be AP — if a rating is 2 seconds stale during a regional network blip, it's a non-issue and the app should still load."

---

## 5. Internal Working (step-by-step)

**Normal operation (no partition):**
1. Client sends write to a coordinator node.
2. Coordinator forwards to **W** replicas (write quorum).
3. Once W replicas ack, client gets success.
4. Reads go to **R** replicas, coordinator returns the most recent version (using timestamps/version vectors).

**When a partition happens:**
5. Nodes detect the partition via **heartbeat/gossip protocol timeouts** (no response within X ms → assume unreachable).
6. Cluster splits into groups (e.g., majority side, minority side).

**If the system is CP:**
7. The side that **cannot reach quorum** (usually the minority) **stops accepting writes** (sometimes reads too) and returns errors — sacrificing availability to avoid serving/accepting inconsistent data.
8. Once the partition heals, minority side catches up via replication/log shipping.

**If the system is AP:**
7. **Both sides keep accepting writes/reads independently.**
8. Conflicting writes are tracked using **vector clocks / version numbers / last-write-wins (LWW) timestamps**.
9. Uses tricks like **hinted handoff** (a healthy node temporarily stores writes meant for a down node) and **sloppy quorum** (accept writes from any available node, not strictly the "correct" replica set).
10. Once partition heals, **anti-entropy repair / read-repair** reconciles the conflicting versions (merge, LWW, or push conflict to application layer — e.g., shopping cart merge logic).

This flow (quorum, heartbeat detection, hinted handoff, anti-entropy repair) is exactly what's asked when interviewers say **"how does Cassandra/DynamoDB actually resolve this under the hood?"**

---

## 6. Most Common Interview Questions

**Q1: What is CAP theorem, in your own words?**
> A: In a distributed system, when a network partition occurs, you can only guarantee either consistency or availability, not both. Partition tolerance itself isn't optional in real systems — it's a given.
- Follow-up: "Why isn't CA possible in practice?" → Because any real distributed system will face network partitions eventually; CA only holds for a single node or a system that assumes network never fails.
- Common mistake: Saying "pick any 2 of 3" without acknowledging P is mandatory in real systems — instantly flags you as someone who memorized a diagram, not the concept.

**Q2: Give examples of CP and AP databases.**
> A: CP — MongoDB (majority write concern), HBase, Zookeeper, etcd. AP — Cassandra, DynamoDB, Riak.
- Follow-up: "Where would Spanner fit?" → Google Spanner tries to be both, using synchronized atomic clocks (TrueTime) to minimize the consistency-availability tradeoff — worth mentioning to show depth, but don't over-explain unless asked.

**Q3: CAP's "C" vs ACID's "C" — same thing?**
> A: No. ACID consistency = data obeys defined constraints/invariants (e.g., foreign keys, business rules). CAP consistency = linearizability — every read sees the latest write. They're unrelated concepts that happen to share a letter.
- Common mistake: Conflating the two — a huge red flag for interviewers.

**Q4: How would you design an eventually-consistent system so it "feels" consistent to the user?**
> A: Techniques: read-your-own-writes (route a user's reads to the node they wrote to), sticky sessions, client-side version tracking, or "read repair" on access.
- Follow-up: "What if two users edit the same cart from different devices?" → Talk about conflict resolution: LWW, vector clocks, or app-level merge (classic Amazon shopping cart example).

**Q5: Is CAP theorem still relevant, given PACELC exists?**
> A: CAP explains partition-time tradeoffs; PACELC extends it to also cover the "no partition" case — the latency vs. consistency tradeoff that exists even during normal operation. Both are relevant; PACELC is just more complete. Bringing this up unprompted signals seniority.

**Q6: Design a highly available service that also needs some consistency guarantees (e.g., a booking/seat-reservation system).**
> A: This is the classic "trap" question. Good answer: don't pick a single global mode. Use **CP for the critical write path** (seat lock/booking commit — often via a single-writer resource or distributed lock like Redis/Zookeeper) and **AP for read-heavy, non-critical paths** (seat map display, search). Mention idempotency keys to handle retries safely.
- Common mistake: Trying to make the entire system either fully CP or fully AP — real systems mix both per component.

---

## 7. Senior-Level Understanding

**Tradeoffs**
- **CP** → safer data, but during a partition, part of your system goes **down or errors out**. Users see failures instead of stale data.
- **AP** → system stays up, but you now own the complexity of **conflict resolution**, and users may see stale/inconsistent data.

**When NOT to force strong consistency**
- Anything read-heavy and tolerant of staleness: feeds, recommendations, view/like counters, catalog browsing.
- Forcing strong consistency here just adds latency and reduces availability for no real business benefit.

**When you must insist on consistency**
- Money movement, inventory decrement, seat/ticket booking, auth/session invalidation, distributed locks.

**Performance implications**
- Strong consistency = synchronous replication / quorum reads-writes = **higher latency**, especially cross-region.
- This is why globally distributed systems (multi-region) almost always lean AP for non-critical data — cross-region synchronous writes can add 100s of ms.

**Scalability implications**
- AP systems scale horizontally much more easily (no coordination bottleneck).
- CP systems often bottleneck on the leader/quorum coordination — this is why systems like etcd/Zookeeper are used for **small, critical metadata**, not high-throughput data.

**Common production issues**
- **Split-brain**: two nodes both think they're the leader after a partition — can cause double-writes/corruption. Root cause of many real outages.
- **Stale leader**: client caches an old leader/service pointer, keeps sending writes to the wrong node.
- **Clock skew**: breaks last-write-wins conflict resolution if node clocks aren't synced (NTP drift) — can silently drop the "newer" write.
- **Cascading retries**: when a CP system starts rejecting writes during a partition, naive clients retry aggressively, worsening the overload once the partition heals ("thundering herd on recovery").

---

## 8. Comparison Section

### CP vs AP Databases
| | CP Systems | AP Systems |
|---|---|---|
| Examples | MongoDB, HBase, Zookeeper, etcd, Redis Cluster | Cassandra, DynamoDB, Riak, CouchDB (configurable) |
| Behavior on partition | Minority side stops serving to stay correct | Both sides keep serving, reconcile later |
| Best for | Financial data, locks, config/metadata, leader election | High write throughput, multi-region, feeds, logs, IoT |
| Latency | Higher (needs quorum/majority) | Lower (tunable, often async) |
| Conflict handling | Avoided by design (only one side is "live") | Needed (vector clocks, LWW, app-level merge) |

### CAP vs ACID vs BASE
| | CAP | ACID | BASE |
|---|---|---|---|
| Scope | Distributed systems tradeoff | Single transaction guarantees | Distributed system philosophy (opposite of ACID) |
| Focus | Consistency vs Availability during partition | Atomicity, Consistency, Isolation, Durability | Basically Available, Soft state, Eventually consistent |
| Typical use | Choosing DB/architecture | Relational DBs, single-node transactions | NoSQL / AP-style systems |

### PACELC Summary
| Condition | Tradeoff |
|---|---|
| Partition (P) | Availability (A) vs Consistency (C) |
| Else (no partition) (E) | Latency (L) vs Consistency (C) |

---

## 9. Production Debugging / Failure Cases

**Case 1: Split-brain in a Zookeeper/etcd cluster**
- Symptom: two services both believe they're the primary, both accepting writes.
- Debug: check quorum membership logs, election term numbers, node heartbeat gaps in monitoring.
- Fix/prevention: proper quorum sizing (odd number of nodes, e.g., 3 or 5), fencing tokens so a demoted leader physically can't write.

**Case 2: Stale read causing double-booking (ticket/seat system)**
- Symptom: two users booked the same seat because a read hit a lagging replica.
- Debug: check **replication lag metrics** (e.g., Mongo oplog lag, Cassandra hinted-handoff backlog).
- Fix: route booking-critical reads to the primary / use quorum reads (`R` high enough) for that specific operation, keep eventual consistency only for browsing.

**Case 3: Cross-region write latency spike mistaken for an outage**
- Symptom: writes timing out across DCs, alerts firing.
- Debug: distributed tracing to see where the request is stalling (network hop vs. quorum wait vs. DB lock).
- Monitoring to have in place: replication lag dashboard, per-node heartbeat/gossip health, quorum-write success rate, error rate segmented by region.

**What to monitor for CAP-related issues in production**
- Replication lag (seconds behind primary)
- Quorum/majority write success rate
- Leader election frequency (frequent re-elections = flaky network)
- Read/write error rates during network blips
- Conflict/merge counts (for AP systems doing LWW or vector-clock resolution)

---

## 10. Important Code Examples

### A) Quorum read/write logic (simplified, to show you understand the mechanism — not a toy "hello world")

```python
class ReplicaSet:
    def __init__(self, replicas: list, replication_factor: int):
        self.replicas = replicas          # list of Node objects
        self.n = replication_factor

    def write(self, key, value, w: int):
        """Write succeeds only if W replicas acknowledge."""
        acks = 0
        for node in self.replicas[:self.n]:
            if node.is_reachable():
                node.store(key, value, version=self._new_version())
                acks += 1
            if acks >= w:
                return {"status": "success", "acked_by": acks}
        return {"status": "failed", "reason": "quorum not reached", "acked_by": acks}

    def read(self, key, r: int):
        """Read from R replicas, return the highest-version value."""
        responses = []
        for node in self.replicas[:self.n]:
            if node.is_reachable():
                responses.append(node.fetch(key))
            if len(responses) >= r:
                break
        if not responses:
            return {"status": "failed", "reason": "quorum not reached"}
        latest = max(responses, key=lambda r: r["version"])
        return {"status": "success", "value": latest["value"]}
```
Interview point to say out loud: *"If W + R > N here, every read is guaranteed to see the latest committed write — that's how I'd explain tunable consistency in Cassandra/DynamoDB."*

### B) Real-world usage — setting consistency level per query (Cassandra driver style)

```python
from cassandra.cluster import Cluster
from cassandra.query import SimpleStatement, ConsistencyLevel

cluster = Cluster(["10.0.0.1", "10.0.0.2", "10.0.0.3"])
session = cluster.connect("orders_keyspace")

# AP-style: fast, tolerant of staleness — fine for a product view counter
fast_query = SimpleStatement(
    "SELECT * FROM product_views WHERE product_id=%s",
    consistency_level=ConsistencyLevel.ONE
)

# CP-style: slower but guarantees latest data — needed for order status
critical_query = SimpleStatement(
    "SELECT status FROM orders WHERE order_id=%s",
    consistency_level=ConsistencyLevel.QUORUM
)
```
This is the kind of code that shows an interviewer you don't just know the theory — you know **where in a real codebase this decision gets made.**

### C) Handling a CP system rejecting requests during a partition (client-side)

```python
import time
import random

def write_with_backoff(client, key, value, max_retries=5):
    for attempt in range(max_retries):
        try:
            return client.write(key, value)
        except QuorumUnavailableError:
            # Don't hammer the cluster right when it's recovering (thundering herd)
            sleep_time = (2 ** attempt) + random.uniform(0, 0.5)
            time.sleep(sleep_time)
    raise Exception("Write failed after retries — surfacing error to caller instead of silently dropping")
```

---

## 11. Revision Notes (Must-Know Priority)

**🔴 Must know cold (asked in almost every system design round):**
- C, A, P definitions — and that **P is not optional** in real systems.
- Real choice is **CP vs AP**, not "pick 2 of 3."
- CAP's "C" ≠ ACID's "C" (linearizability vs. constraints).
- At least 2 real DB examples for CP and 2 for AP.
- One concrete production example of choosing CP vs AP in the same system (e.g., payments = CP, catalog = AP).

**🟠 Should know (senior/staff level differentiation):**
- PACELC and why it's a more complete model.
- Quorum math: W + R > N.
- Split-brain, hinted handoff, anti-entropy repair, vector clocks/LWW.
- Read-your-own-writes trick for making AP systems "feel" consistent.

**🟡 Good to mention if it comes up naturally (shows depth, don't force it):**
- Google Spanner's approach (TrueTime) as an attempt to minimize the tradeoff.
- Clock skew breaking LWW resolution.
- Idempotency keys for safe retries in CP systems during partition recovery.

**Interview delivery tip:** Don't recite definitions first. Start with *"it depends on what the data is used for"* → give the payments-vs-catalog example → then formalize with CP/AP/PACELC. That ordering alone signals production experience over textbook memorization.
